# Qwen3.5 검수 전 100 step 비교 실험 v12
작성자 김헌영 · 2026-10-08

기존 20 step 실험과 동일한 학습112개·검증28개, 원본·프롬프트·모델 commit·토큰 매핑을 사용합니다. **기본 모델에서 새로 총100 step 학습**하며 이전 어댑터를 이어서 학습하지 않습니다. 학습률은 동일한 초기값2e-4와 선형 감소를 사용하지만 종료 시점이100step이라 감소 일정은 달라집니다.

검증뿐 아니라 학습 사례의 자유 생성과 첫 위치의 정답 함수/종료 토큰 확률을 기록합니다. 종료 토큰을 강제로 금지하지 않습니다. 사람 검수는 필요하지 않으며 결과는 미검수 초안 정답 기준입니다.

이 폴더의 helper와 노트북을 Drive cropy_qwen_v11/Colab_검수전_100step_v12에 업로드합니다. 이전 런타임의 모델이 남지 않도록 런타임을 새로 시작하고 GPU를 선택한 뒤 순서대로 실행하세요. GPU 실험은 로컬에서 아직 실행하지 않았습니다.


In [ ]:
from pathlib import Path
import sys,json,hashlib
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR=Path('/content/drive/MyDrive/cropy_qwen_v11')
CODE_DIR=DATA_DIR/'Colab_검수전_100step_v12'
SOURCE=DATA_DIR/'Cropy_Qwen35_기능별400_검수대기_v11.xlsx'
MODEL_ID='Qwen/Qwen3.5-2B'  # 점검 후보 크기를 확인하고 수정 가능
MODEL_REF='15852e8c16360a2fea060d615a32b45270f8a8fc'  # 다음 단계에서 실제 commit SHA로 해석 후 고정
RUN_MODEL_STEPS=True
MAX_LENGTH=4096
MAX_STEPS=100
MAX_TRAIN_PER_FUNCTION=16
MAX_EVAL_PER_FUNCTION=4
import datetime
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR=DATA_DIR/('unreviewed_100step_v12_'+RUN_ID)
EXPECTED_SOURCE_SHA256='d86800d67a3f25e09444f6e79ac1247080ff6bef8e5d7b9a14d4d02dd2c79e97'
SEED=10810
DATA_READY=TOKEN_CHECKED=MASK_CHECKED=GRADIENT_CHECKED=False
for p in [SOURCE,CODE_DIR/'cropy_qwen_pilot.py']:
    assert p.is_file(),f'필요한 파일 없음 {p}'

BASELINE_ENV={'torch': '2.11.0+cu130', 'transformers': '5.19.0', 'peft': '0.21.2', 'gpu': 'Tesla T4', 'dtype': 'torch.bfloat16', 'model_id': 'Qwen/Qwen3.5-2B', 'revision': '15852e8c16360a2fea060d615a32b45270f8a8fc', 'source_sha256': 'd86800d67a3f25e09444f6e79ac1247080ff6bef8e5d7b9a14d4d02dd2c79e97', 'review_mode': 'unreviewed_experiment', 'human_approval_required': False, 'labels_human_validated': False, 'seed': 10810, 'train_count': 112, 'eval_count': 28}
BASELINE_SELECTION={'train': ['V5_Q_044_3', 'V5_Q_087_2', 'V5_Q_040_1', 'V5_Q_016_2', 'V5_Q_093_2', 'V5_Q_029_3', 'V5_Q_023_3', 'V5_Q_018_2', 'V5_Q_039_3', 'V5_Q_038_1', 'V5_Q_008_4', 'V5_Q_011_4', 'V5_Q_007_4', 'V5_Q_004_3', 'V5_Q_070_1', 'V5_Q_031_3', 'V5_C_045_2', 'V5_C_074_3', 'V5_C_020_1', 'V5_C_064_3', 'V5_C_098_3', 'V5_C_037_3', 'V5_C_096_1', 'V5_C_044_3', 'V5_C_015_3', 'V5_C_038_2', 'V5_C_011_2', 'V5_C_030_1', 'V5_C_084_3', 'V5_C_018_3', 'V5_C_090_3', 'V5_C_006_2', 'STT_016', 'V5_D_040_3', 'V5_D_046_3', 'V5_D_056_2', 'V5_D_064_2', 'V5_D_094_2', 'V5_D_040_2', 'V5_D_042_1', 'V5_D_026_2', 'V5_D_065_1', 'V5_D_021_1', 'V5_D_030_2', 'V5_D_018_1', 'V5_D_014_1', 'V5_D_062_1', 'V5_D_069_1', 'P400_367', 'STT_032', 'P400_091', 'P400_148', 'P400_164', 'P400_359', 'P400_355', 'P400_209', 'P400_014', 'P400_230', 'P400_198', 'P400_295', 'P400_320', 'P400_338', 'P400_319', 'P_P005', 'V5_M_004_2', 'V5_M_036_1', 'V5_M_074_1', 'V5_M_078_3', 'V5_M_029_2', 'V5_M_037_2', 'V5_M_020_1', 'V5_M_017_3', 'V5_M_045_1', 'V5_M_074_2', 'V5_M_056_1', 'V5_Q_021_1', 'V5_M_054_1', 'V5_M_001_2', 'V5_M_035_1', 'V5_M_018_1', 'V5_R_055_3', 'V5_R_026_2', 'V5_R_056_3', 'V5_R_084_1', 'V5_R_002_2', 'V5_R_066_1', 'V5_R_056_2', 'V5_R_078_2', 'V5_R_035_2', 'V5_R_075_1', 'V5_R_096_3', 'V5_R_015_3', 'V5_R_024_1', 'V5_R_076_2', 'V5_R_047_2', 'V5_R_023_1', 'V5_S_032_1', 'V5_S_006_4', 'V5_S_080_1', 'V5_S_039_2', 'V5_S_063_2', 'V5_S_013_2', 'V5_S_075_1', 'V5_S_011_3', 'V5_S_022_1', 'V5_S_011_4', 'V5_S_003_4', 'V5_S_065_1', 'V5_S_081_1', 'V5_S_024_2', 'V5_S_001_3', 'V5_S_056_3'], 'validation': ['V5_Q_113_3', 'P_F017', 'V5_Q_101_1', 'V5_Q_113_2', 'V5_C_110_2', 'V5_C_107_2', 'V5_C_101_3', 'V5_C_121_2', 'V5_D_107_2', 'V5_D_103_3', 'V5_D_127_2', 'V5_D_110_1', 'P400_377', 'P400_114', 'P400_184', 'P400_342', 'V5_M_129_2', 'V5_M_115_3', 'STT_038', 'V5_M_121_1', 'V5_R_122_1', 'V5_R_104_2', 'V5_R_110_2', 'V5_R_117_3', 'V5_S_124_1', 'P_S010', 'V5_S_111_1', 'STT_040']}
BASELINE_TOKENS={'<hs_get_diagnosis>': 248077, '<hs_topic_result>': 248078, '<hs_topic_severity>': 248079, '<hs_topic_observation>': 248080, '<hs_topic_provenance>': 248081, '<hs_topic_impact>': 248082, '<hs_topic_food_safety>': 248083, '<hs_topic_change>': 248084, '<hs_get_prediction_metrics>': 248085, '<hs_topic_confidence>': 248086, '<hs_topic_ranking>': 248087, '<hs_topic_distribution>': 248088, '<hs_topic_limits>': 248089, '<hs_topic_comparison>': 248090, '<hs_get_recapture_guidance>': 248091, '<hs_topic_where>': 248092, '<hs_topic_why>': 248093, '<hs_topic_certainty>': 248094, '<hs_get_care_information>': 248095, '<hs_topic_causes>': 248096, '<hs_topic_prevention>': 248097, '<hs_topic_priority>': 248098, '<hs_get_pesticide_information>': 248099, '<hs_topic_list>': 248100, '<hs_topic_dilution>': 248101, '<hs_topic_usage>': 248102, '<hs_topic_phi>': 248103, '<hs_topic_reentry>': 248104, '<hs_topic_verification>': 248105, '<hs_topic_ingredient>': 248106, '<hs_topic_summary>': 248107, '<hs_topic_checklist>': 248108, '<hs_topic_mixing>': 248109, '<hs_clarify_request>': 248110, '<hs_target_crop>': 248111, '<hs_target_disease>': 248112, '<hs_target_topic>': 248113, '<hs_target_confidence>': 248114, '<hs_target_severity>': 248115, '<hs_target_record>': 248116, '<hs_target_product>': 248117, '<hs_target_stt>': 248118, '<hs_handle_scope_limit>': 248119, '<hs_reason_crop>': 248120, '<hs_reason_action>': 248121, '<hs_reason_input>': 248122, '<hs_product_current>': 248123, '<hs_product_all>': 248124, '<hs-end>': 248125}
BASELINE_SYSTEM_SHA256='395476f68209dc6b92e26963bae162a8ca441e3a6bb87b7f3a9c9fb053d9a7c7'
BASELINE_EVAL={'format_valid': 0.0, 'end_valid': 1.0, 'function_correct': 0.0, 'arguments_correct': 0.0, 'exact_correct': 0.0, 'per_function': {'get_diagnosis': {'support': 4, 'function_accuracy': 0.0}, 'get_prediction_metrics': {'support': 4, 'function_accuracy': 0.0}, 'get_recapture_guidance': {'support': 4, 'function_accuracy': 0.0}, 'get_care_information': {'support': 4, 'function_accuracy': 0.0}, 'get_pesticide_information': {'support': 4, 'function_accuracy': 0.0}, 'clarify_request': {'support': 4, 'function_accuracy': 0.0}, 'handle_scope_limit': {'support': 4, 'function_accuracy': 0.0}}, 'macro_function_accuracy': 0.0, 'review_mode': 'unreviewed_experiment', 'labels_human_validated': False}


## 1 미검수 초안의 형식·분할 검사
원본 해시, 함수·인자·토큰 형식, AI 검토 상태, 분할 교차, 기능별 320/80을 검사합니다. 사람 승인 검사는 이 실험에서 요구하지 않습니다.

In [ ]:
import importlib.util,subprocess,importlib
if importlib.util.find_spec('openpyxl') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','openpyxl'])
sys.path.insert(0,str(CODE_DIR))
import cropy_qwen_pilot as helper
helper=importlib.reload(helper)
assert Path(helper.__file__).resolve()==(CODE_DIR/'cropy_qwen_pilot.py').resolve(),'다른 폴더의 helper가 로드됐습니다 런타임 재시작 필요'
DATA_READY=False
cases,schemas,tokens,system=helper.load_experimental_dataset(SOURCE,EXPECTED_SOURCE_SHA256)
assert hashlib.sha256(system.encode()).hexdigest()==BASELINE_SYSTEM_SHA256
lookup={r['conversation_id']:r for r in cases}
train_rows=[lookup[cid] for cid in BASELINE_SELECTION['train']]
eval_rows=[lookup[cid] for cid in BASELINE_SELECTION['validation']]
assert len(train_rows)==112 and len(eval_rows)==28
assert all(r['split']=='train' for r in train_rows) and all(r['split']=='validation' for r in eval_rows)
assert set(BASELINE_SELECTION['train']).isdisjoint(BASELINE_SELECTION['validation'])
from collections import Counter
assert Counter(r['function_name'] for r in train_rows)==Counter({fn:16 for fn in schemas})
assert Counter(r['function_name'] for r in eval_rows)==Counter({fn:4 for fn in schemas})
DATA_READY=True
print('기존 실험과 동일한 학습112개 검증28개 선택 확인')


## 2 이전 실행의 패키지와 환경
Transformers·PEFT·Accelerate·bitsandbytes·tokenizers는 이전 버전으로 설치합니다. PyTorch와 GPU 환경은 실제 기록하고 이전 환경과 비교합니다. 패키지 설치 후 런타임 재시작 안내가 나오면 처음부터 다시 실행하세요.

In [ ]:
assert DATA_READY,'데이터 형식과 분할 검사를 먼저 완료하세요'
assert RUN_MODEL_STEPS,'모델 크기를 확인한 후 설정에서 RUN_MODEL_STEPS=True로 변경하세요'
subprocess.check_call([sys.executable,'-m','pip','install','-U','transformers==5.19.0','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2','tokenizers==0.23.2'])
import torch,transformers,peft,inspect,re,random
from huggingface_hub import HfApi
assert torch.cuda.is_available(),'GPU 런타임을 선택하세요'
random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR.mkdir(parents=True,exist_ok=False) # 기존 실험을 덮어쓰지 않음 새 실행은 새 경로 설정
REVISION=HfApi().model_info(MODEL_ID,revision=MODEL_REF).sha
assert re.fullmatch(r'[0-9a-f]{40}',REVISION)
DTYPE=getattr(torch,BASELINE_ENV['dtype'].split('.')[-1])
versions={'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,'gpu':torch.cuda.get_device_name(0),'dtype':str(DTYPE),'model_id':MODEL_ID,'revision':REVISION,'source_sha256':helper.sha256(SOURCE),'review_mode':'unreviewed_experiment','human_approval_required':False,'labels_human_validated':False,'seed':SEED,'train_count':len(train_rows),'eval_count':len(eval_rows),'max_steps':MAX_STEPS,'initial_learning_rate':2e-4,'gradient_accumulation_steps':4,'base_restart':True}
(OUTPUT_DIR/'environment.json').write_text(json.dumps(versions,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print(versions)

(OUTPUT_DIR/'selected_cases.json').write_text(json.dumps({'train':[r['conversation_id'] for r in train_rows],'validation':[r['conversation_id'] for r in eval_rows]},ensure_ascii=False,indent=2),encoding='utf-8')

env_differences={k:{"baseline":BASELINE_ENV[k],"candidate":versions[k]} for k in ["torch","transformers","peft","gpu","dtype","model_id","revision","seed","source_sha256"] if BASELINE_ENV[k]!=versions[k]}
(OUTPUT_DIR/"environment_comparison.json").write_text(json.dumps({"differences":env_differences,"recorded_fields_equal":not env_differences},ensure_ascii=False,indent=2),encoding="utf-8")
print("기록된 환경 차이",env_differences)


## 3 49개 토큰 등록 및 정답 구간 검사
Qwen 기본 EOS를 유지하고 hs-end에서 별도로 중단합니다. 프롬프트에 포함된 빈 think 영역을 모델 출력으로 평가하지 않습니다. 전체 assistant 메시지를 템플릿에 재직렬화하지 않고 생성 프롬프트와 hs completion을 분리합니다.

In [ ]:
assert DATA_READY and RUN_MODEL_STEPS
from transformers import AutoTokenizer
TOKEN_CHECKED=MASK_CHECKED=False
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
BASE_EOS=tokenizer.eos_token_id
assert BASE_EOS is not None and tokenizer.chat_template
params=inspect.signature(tokenizer.add_special_tokens).parameters
if 'replace_extra_special_tokens' in params:
    tokenizer.add_special_tokens({'extra_special_tokens':tokens},replace_extra_special_tokens=False)
else:
    tokenizer.add_special_tokens({'additional_special_tokens':tokens},replace_additional_special_tokens=False)
ids={s:tokenizer.encode(s,add_special_tokens=False) for s in tokens}
assert len(tokens)==49 and all(len(x)==1 for x in ids.values())
token_ids={s:v[0] for s,v in ids.items()}
assert len(set(token_ids.values()))==49 and BASE_EOS not in token_ids.values()
assert tokenizer.eos_token_id==BASE_EOS
for text,i in token_ids.items():assert tokenizer.decode([i],skip_special_tokens=False)==text
END_ID=token_ids['<hs-end>']
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
TOKEN_CHECKED=True
train_encoded=[helper.encode_case(tokenizer,r,system,MAX_LENGTH) for r in train_rows]
eval_encoded=[helper.encode_case(tokenizer,r,system,MAX_LENGTH) for r in eval_rows]
for x,r in zip(train_encoded+eval_encoded,train_rows+eval_rows):
    pos=next(i for i,v in enumerate(x['labels']) if v!=-100)
    assert all(v==-100 for v in x['labels'][:pos])
    assert x['labels'][pos:]==tokenizer.encode(r['llm_output'],add_special_tokens=False)
    assert x['labels'][-1]==END_ID and x['labels'][pos] in token_ids.values()
    assert x['input_ids'][pos:]==x['labels'][pos:]
MASK_CHECKED=True
(OUTPUT_DIR/'token_manifest.json').write_text(json.dumps({'tokens':token_ids,'base_eos':BASE_EOS,'model_id':MODEL_ID,'base_revision':REVISION,'system_sha256':hashlib.sha256(system.encode()).hexdigest()},ensure_ascii=False,indent=2),encoding='utf-8')
print('단일 토큰49 및 loss mask 확인',max(len(x['input_ids']) for x in train_encoded+eval_encoded))

assert token_ids==BASELINE_TOKENS,"기존 실행과 토큰 ID가 다릅니다"


## 4 모델 로딩과 LoRA
4bit는 GPU 메모리를 줄이는 파일럿 로딩 방식입니다. 보드용 양자화 결과가 아닙니다. 텍스트 영역의 실제 q_proj·v_proj만 선택하고 새 토큰 임베딩도 학습합니다. 선택 모델이나 라이브러리에서 지원하지 않으면 중단합니다.

In [ ]:
assert DATA_READY and RUN_MODEL_STEPS and TOKEN_CHECKED and MASK_CHECKED
from transformers import BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel
MODEL_CLASS=getattr(transformers,'Qwen3_5ForConditionalGeneration',None)
assert MODEL_CLASS is not None,'현재 transformers가 Qwen3.5 full config를 지원하지 않습니다'
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=DTYPE)
def load_base():
    m=MODEL_CLASS.from_pretrained(MODEL_ID,revision=REVISION,quantization_config=quant,device_map={'':0},dtype=DTYPE,attn_implementation='sdpa')
    m.resize_token_embeddings(max(len(tokenizer),m.get_input_embeddings().num_embeddings))
    assert max(token_ids.values())<m.get_input_embeddings().num_embeddings
    assert m.get_input_embeddings().weight.data_ptr()==m.get_output_embeddings().weight.data_ptr(),'이 파일럿은 tied embedding 경로만 지원함'
    return m
base=load_base()
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
embed_name=next(name for name,module in base.named_modules() if module is base.get_input_embeddings())
text_root=embed_name.rsplit('.',1)[0]
targets=[name for name,module in base.named_modules() if name.startswith(text_root+'.') and name.rsplit('.',1)[-1] in {'q_proj','v_proj'} and isinstance(module,torch.nn.Linear)]
assert targets and all('visual' not in name and 'vision' not in name for name in targets)
assert 'trainable_token_indices' in inspect.signature(LoraConfig).parameters and 'ensure_weight_tying' in inspect.signature(LoraConfig).parameters
config=LoraConfig(task_type='CAUSAL_LM',r=8,lora_alpha=16,lora_dropout=0.05,target_modules=targets,trainable_token_indices={embed_name:list(token_ids.values())},ensure_weight_tying=True)
model=get_peft_model(base,config)
model.config.use_cache=False
model.print_trainable_parameters()
print('텍스트 LoRA 대상',targets,'추가 토큰 대상',embed_name)


## 5 gradient 소량 확인과 학습
먼저 한 사례의 backward로 추가 토큰 gradient를 검사하고 지웁니다. 이 확인에서는 optimizer step을 수행하지 않습니다. 다음 Trainer 단계만 실제 파일럿 학습입니다.

In [ ]:
assert DATA_READY and TOKEN_CHECKED and MASK_CHECKED and RUN_MODEL_STEPS
GRADIENT_CHECKED=False
class Rows(torch.utils.data.Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):return self.rows[i]
def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    return {k:torch.tensor([r[k]+[(-100 if k=='labels' else tokenizer.pad_token_id if k=='input_ids' else 0)]*(longest-len(r[k])) for r in rows],dtype=torch.long) for k in ['input_ids','attention_mask','labels']}
model.train();device=model.get_input_embeddings().weight.device
batch={k:v.to(device) for k,v in collate([train_encoded[0]]).items()}
loss=model(**batch).loss
assert torch.isfinite(loss)
loss.backward()
token_grads=[p.grad for name,p in model.named_parameters() if 'trainable_tokens' in name and p.requires_grad]
assert token_grads and all(g is not None and torch.isfinite(g).all() for g in token_grads)
assert any(g.abs().sum().item()>0 for g in token_grads)
model.zero_grad(set_to_none=True);del batch,loss,token_grads
GRADIENT_CHECKED=True
from transformers import Trainer,TrainingArguments
args=TrainingArguments(output_dir=str(OUTPUT_DIR/'trainer'),max_steps=MAX_STEPS,per_device_train_batch_size=1,gradient_accumulation_steps=4,learning_rate=2e-4,bf16=DTYPE==torch.bfloat16,fp16=DTYPE==torch.float16,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},logging_steps=1,save_strategy='steps',save_steps=20,save_total_limit=2,report_to='none',seed=SEED,remove_unused_columns=False,optim='adamw_torch')
trainer=Trainer(model=model,args=args,train_dataset=Rows(train_encoded),data_collator=collate)
assert GRADIENT_CHECKED
train_result=trainer.train()
assert trainer.state.global_step==MAX_STEPS
trainer.save_state()
(OUTPUT_DIR/'train_metrics.json').write_text(json.dumps({**train_result.metrics,'review_mode':'unreviewed_experiment','labels_human_validated':False},indent=2),encoding='utf-8')


## 5 학습·검증 자유 생성 및 첫 토큰 진단
100step 후 같은 학습112개와 검증28개를 평가합니다. 정답 첫 토큰과 hs-end의 확률을 프롬프트 끝 위치에서 기록합니다. 종료만 출력하는 비율과 학습/검증 차이를 확인합니다. 생성 토큰을 강제로 제한하지 않습니다.

In [ ]:
assert DATA_READY and RUN_MODEL_STEPS and GRADIENT_CHECKED

def evaluate(m,rows,encoded,first_token_diagnostics=True):
    assert len(rows)==len(encoded) and rows
    m.eval();results=[]
    for row,x in zip(rows,encoded):
        pos=next(i for i,y in enumerate(x['labels']) if y!=-100)
        dev=m.get_input_embeddings().weight.device
        prompt=torch.tensor([x['input_ids'][:pos]],device=dev)
        with torch.no_grad():
            out=m.generate(input_ids=prompt,attention_mask=torch.ones_like(prompt),max_new_tokens=8,do_sample=False,eos_token_id=[END_ID,BASE_EOS],pad_token_id=tokenizer.pad_token_id,use_cache=True)[0,pos:].tolist()
        result=helper.score_generation(out,row,token_ids,schemas)
        if first_token_diagnostics:
            with torch.no_grad():
                logits=m(input_ids=prompt,attention_mask=torch.ones_like(prompt),use_cache=False,logits_to_keep=1).logits[0,-1].float()
                probs=torch.softmax(logits,dim=-1)
                gold=x['labels'][pos];best=int(logits.argmax().item())
                result['first_position']={'expected_id':gold,'expected_probability':float(probs[gold].item()),'end_probability':float(probs[END_ID].item()),'expected_rank':int((logits>logits[gold]).sum().item())+1,'argmax_id':best,'argmax_token':next((s for s,i in token_ids.items() if i==best),f'UNKNOWN_ID_{best}')}
            del logits,probs
        results.append(result)
    keys=['format_valid','end_valid','function_correct','arguments_correct','exact_correct','first_token_correct','end_only']
    summary={k:sum(r[k] for r in results)/len(results) for k in keys}
    summary['support']=len(results)
    summary['per_function']={fn:{'support':sum(r['expected_function']==fn for r in results),'function_accuracy':sum(r['function_correct'] for r in results if r['expected_function']==fn)/sum(r['expected_function']==fn for r in results),'exact_accuracy':sum(r['exact_correct'] for r in results if r['expected_function']==fn)/sum(r['expected_function']==fn for r in results)} for fn in schemas}
    summary['macro_function_accuracy']=sum(v['function_accuracy'] for v in summary['per_function'].values())/len(schemas)
    if first_token_diagnostics:
        summary['mean_expected_first_probability']=sum(r['first_position']['expected_probability'] for r in results)/len(results)
        summary['mean_end_first_probability']=sum(r['first_position']['end_probability'] for r in results)/len(results)
    summary.update(review_mode='unreviewed_experiment',labels_human_validated=False)
    return results,summary

train_generation,train_generation_metrics=evaluate(model,train_rows,train_encoded)
(OUTPUT_DIR/'train_generation_cases.json').write_text(json.dumps(train_generation,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'train_generation_metrics.json').write_text(json.dumps(train_generation_metrics,ensure_ascii=False,indent=2),encoding='utf-8')
before_reload,metrics=evaluate(model,eval_rows,eval_encoded)
(OUTPUT_DIR/'eval_cases.json').write_text(json.dumps(before_reload,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'eval_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
comparison={'baseline_steps':20,'candidate_steps':MAX_STEPS,'same_case_ids':True,'same_source_sha256':True,'labels_human_validated':False,'recorded_environment_differences':env_differences,'metrics':{k:{'baseline':BASELINE_EVAL[k],'candidate':metrics[k],'delta_pp':100*(metrics[k]-BASELINE_EVAL[k])} for k in ['format_valid','end_valid','function_correct','arguments_correct','exact_correct']}}
(OUTPUT_DIR/'comparison_20_vs_100.json').write_text(json.dumps(comparison,ensure_ascii=False,indent=2),encoding='utf-8')
ADAPTER_DIR=OUTPUT_DIR/'adapter'
model.save_pretrained(ADAPTER_DIR,safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
del trainer,model,base
torch.cuda.empty_cache()
loaded_tok=AutoTokenizer.from_pretrained(ADAPTER_DIR)
assert {s:loaded_tok.encode(s,add_special_tokens=False)[0] for s in tokens}==token_ids
assert all(len(loaded_tok.encode(s,add_special_tokens=False))==1 for s in tokens)
assert loaded_tok.eos_token_id==BASE_EOS and loaded_tok.chat_template==tokenizer.chat_template
assert [helper.encode_case(loaded_tok,r,system,MAX_LENGTH) for r in eval_rows]==eval_encoded
reloaded_base=load_base()
reloaded=PeftModel.from_pretrained(reloaded_base,ADAPTER_DIR,is_trainable=False)
after_reload,_=evaluate(reloaded,eval_rows,eval_encoded,first_token_diagnostics=False)
assert [r['output_ids'] for r in before_reload]==[r['output_ids'] for r in after_reload],'재로딩 출력 불일치'
(OUTPUT_DIR/'reload_check.json').write_text(json.dumps({'token_ids_equal':True,'greedy_outputs_equal':True,'validation_cases':len(after_reload),'quantization':'4bit Colab loading only not board export'}),encoding='utf-8')
print('학습 사례 생성 결과',train_generation_metrics)
print('검증 결과',metrics)
print('20step 대비',comparison)
print('저장 위치',OUTPUT_DIR)


## 해석
학습·검증에서 end_only가 모두 높으면 종료 편향이나 학습 경로를 점검합니다. 학습 사례는 맞추는데 검증만 실패하면 일반화와 데이터 구성을 점검합니다. first_position의 expected_probability, end_probability, expected_rank로 정답 함수와 종료 토큰의 경쟁을 확인할 수 있습니다.

20step과100step은 같은 초기 학습률과 선형 schedule을 쓰지만 종료 시점이 달라 학습률 경로가 같지는 않습니다. PyTorch·GPU 등이 바뀌면 환경 비교 기록에도 차이가 남습니다. 소량·단일 seed·미검수 정답에 대한 비교이며 서비스 성능을 보증하지 않습니다.
